## ✅ Import Libraries

In [1]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import pandas as pd
import joblib
import os
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

## ✅ Loading and Cleaning Data

In [2]:
# Load the cleaned dataset
df = pd.read_csv("../data/processed/cleaned_news.csv")
df["clean_content"] = df["clean_content"].fillna("")

# Define features and target variable
X = df["clean_content"]
y = df["label"]

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Initialize your vectorizer
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))

# Fit and transform
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


## ✅Logistic Regression Model

In [3]:
# 1. define the model
log_reg = LogisticRegression(solver="liblinear",random_state=42)

# 2. Define the parameter distribution
# 'C': Controls regularization strength (smaller = stronger regularization)
# 'penalty': 'l1' (Lasso) vs 'l2' (Ridge)
param_distributions = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2']
}

# 3. Setup RandomizedSearchCV
# n_iter=10: Tries 10 random combinations from the grid above
# cv=5: Uses 5-fold cross-validation
random_search_lr = RandomizedSearchCV(
    estimator=log_reg,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    verbose=1,
    n_jobs=1,
    random_state=42
)

In [4]:
# 4. Fit the search to your data
random_search_lr.fit(X_train_tfidf, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,estimator,LogisticRegre...r='liblinear')
,param_distributions,"{'C': [0.01, 0.1, ...], 'penalty': ['l1', 'l2']}"
,n_iter,10
,scoring,None
,n_jobs,1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [5]:
# 5. Output the results
print(f"Best Parameters: {random_search_lr.best_params_}")
print(f"Best Cross-Validation Score: {random_search_lr.best_score_:.4f}")

# 6. Evaluate on Test Set using the optimized model
best_log_reg = random_search_lr.best_estimator_
y_pred = best_log_reg.predict(X_test_tfidf)
print("\nClassification Report (Optimized):")
print(classification_report(y_test, y_pred))

Best Parameters: {'penalty': 'l2', 'C': 10}
Best Cross-Validation Score: 0.9629

Classification Report (Optimized):
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      5778
           1       0.97      0.97      0.97      6958

    accuracy                           0.96     12736
   macro avg       0.96      0.96      0.96     12736
weighted avg       0.96      0.96      0.96     12736



## ✅Random Forest Model

In [6]:
# 1. Define the model
rf = RandomForestClassifier(random_state=42)

# 2. Define the parameter distribution
# n_estimators: Number of trees (too high = slow, too low = underfit)
# max_depth: Limits how deep trees can grow (crucial to prevent overfitting)
# min_samples_split: Minimum samples required to split a node
param_distributions = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}

# 3. Setup RandomizedSearchCV
# n_iter=10: Tries 10 random combinations
# cv=5: Uses 5-fold cross-validation
random_search_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    verbose=1,
    n_jobs=1,
    random_state=42
)

# 4. Fit the search to your data
random_search_rf.fit(X_train_tfidf, y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/home/home/.pyenv/versions/3.10.13/envs/Fatocheck/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [10, 20], 'min_samples_split': [2, 5], 'n_estimators': [50, 100]}"
,n_iter,10
,scoring,None
,n_jobs,1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [7]:
# 5. Output the results
print(f"Best Parameters: {random_search_rf.best_params_}")
print(f"Best Cross-Validation Score: {random_search_rf.best_score_:.4f}")

# 6. Evaluate on Test Set
best_rf = random_search_rf.best_estimator_
y_pred = best_rf.predict(X_test_tfidf)
print("\nClassification Report (Optimized Random Forest):")
print(classification_report(y_test, y_pred))

Best Parameters: {'n_estimators': 100, 'min_samples_split': 5, 'max_depth': 20}
Best Cross-Validation Score: 0.9383

Classification Report (Optimized Random Forest):
              precision    recall  f1-score   support

           0       0.95      0.93      0.94      5778
           1       0.94      0.96      0.95      6958

    accuracy                           0.94     12736
   macro avg       0.94      0.94      0.94     12736
weighted avg       0.94      0.94      0.94     12736



## ✅XGBoost Model

In [8]:
# 1. Initialize XGBoost classifier
# CPU-based configuration for better WSL stability

xgb_clf = xgb.XGBClassifier(
    tree_method='hist',
    random_state=42,
    eval_metric='logloss'
)

# 2. Define parameter distributions
# Balanced search space for NLP TF-IDF classification

param_distributions = {
    'n_estimators': [100, 200, 300],

    'learning_rate': [0.01, 0.05, 0.1],

    'max_depth': [3, 5, 7],

    'subsample': [0.6, 0.8, 1.0],

    'colsample_bytree': [0.6, 0.8, 1.0],

    'gamma': [0, 1, 5]
}

# 3. Setup RandomizedSearchCV
# n_jobs=1 prevents WSL memory crashes during tuning

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_clf,

    param_distributions=param_distributions,

    n_iter=10,

    cv=3,

    scoring='f1',

    verbose=2,

    n_jobs=1,

    random_state=42
)

In [9]:
# 4. Fit the search to your data
# XGBoost natively accepts your scipy sparse matrix from the TfidfVectorizer.
random_search_xgb.fit(X_train_tfidf, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


[CV] END colsample_bytree=0.8, gamma=5, learning_rate=0.1, max_depth=5, n_estimators=100, subsample=0.6; total time= 1.3min
[CV] END colsample_bytree=0.8, gamma=5, learning_rate=0.1, max_depth=5, n_estimators=100, subsample=0.6; total time= 1.2min
[CV] END colsample_bytree=0.8, gamma=5, learning_rate=0.1, max_depth=5, n_estimators=100, subsample=0.6; total time= 1.3min
[CV] END colsample_bytree=0.6, gamma=1, learning_rate=0.1, max_depth=5, n_estimators=200, subsample=0.8; total time= 2.2min
[CV] END colsample_bytree=0.6, gamma=1, learning_rate=0.1, max_depth=5, n_estimators=200, subsample=0.8; total time= 2.2min
[CV] END colsample_bytree=0.6, gamma=1, learning_rate=0.1, max_depth=5, n_estimators=200, subsample=0.8; total time= 2.2min
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.1, max_depth=3, n_estimators=200, subsample=1.0; total time= 1.2min
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.1, max_depth=3, n_estimators=200, subsample=1.0; total time= 1.2min
[CV] END

,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.6, 0.8, ...], 'gamma': [0, 1, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], ...}"
,n_iter,10
,scoring,'f1'
,n_jobs,1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [10]:
# 5. Output the results
print(f"Best Parameters: {random_search_xgb.best_params_}")
print(f"Best Cross-Validation Score: {random_search_xgb.best_score_:.4f}")

# 6. Evaluate on Test Set using the optimized model
best_xgb = random_search_xgb.best_estimator_
y_pred = best_xgb.predict(X_test_tfidf)
print("\nClassification Report (Optimized XGBoost):")
print(classification_report(y_test, y_pred))

Best Parameters: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.05, 'gamma': 1, 'colsample_bytree': 0.6}
Best Cross-Validation Score: 0.9712

Classification Report (Optimized XGBoost):
              precision    recall  f1-score   support

           0       0.96      0.97      0.97      5778
           1       0.98      0.97      0.97      6958

    accuracy                           0.97     12736
   macro avg       0.97      0.97      0.97     12736
weighted avg       0.97      0.97      0.97     12736



## ✅ Save Tuned Model

In [11]:
# 1. Define a directory to save models
save_dir = "../models/trained"
os.makedirs(save_dir, exist_ok=True)

# 2. Bundle models and vectorizer into pipelines
model_pipelines = {
    'logistic_regression': Pipeline([('tfidf', vectorizer), ('clf', best_log_reg)]),
    'random_forest': Pipeline([('tfidf', vectorizer), ('clf', best_rf)]),
    'xgboost': Pipeline([('tfidf', vectorizer), ('clf', best_xgb)])
}
# 3. Save each pipeline using joblib
for name, pipeline in model_pipelines.items():
    save_path = os.path.join(save_dir, f"{name}_pipeline.joblib")
    joblib.dump(pipeline, save_path)
    print(f"Saved {name} pipeline to {save_path}")

Saved logistic_regression pipeline to ../models/trained/logistic_regression_pipeline.joblib
Saved random_forest pipeline to ../models/trained/random_forest_pipeline.joblib
Saved xgboost pipeline to ../models/trained/xgboost_pipeline.joblib


## ✅ Hyperparameter Tuning Summary

### Key Findings:
**1. Model Performance Champion: XGBoost** 

XGBoost outperformed the other models, achieving the highest cross-validation F1-score (0.9712) and a test accuracy of 97%. It proved highly effective at handling the sparse, high-dimensional matrix generated by the TF-IDF vectorizer.

**2. Performance Comparison**
| Model | Best CV Score | Test Accuracy | Precision (Class 1) | Recall (Class 1) |
| :--- | :--- | :--- | :--- | :--- |
| **XGBoost** | **0.9712** | **0.97** | **0.98** | **0.97** |
| Logistic Regression | 0.9629 | 0.96 | 0.97 | 0.97 |
| Random Forest | 0.9383 | 0.94 | 0.94 | 0.96 |

**3. Architectural Insights**
* **Linear vs. Non-Linear:** Logistic Regression performed exceptionally well (96%), indicating that the TF-IDF features for fake vs. real news are highly linearly separable. 
* **Tree-Based Limitations:** Random Forest struggled slightly compared to the others. Sparse text matrices (10,000 features) often make it difficult for standard Random Forests to find optimal splits without growing excessively deep trees, whereas XGBoost's gradient boosting approach navigated the sparsity efficiently.

**4. Next Steps (Deployment Readiness)**

All three optimized models have been successfully serialized as Scikit-Learn `Pipeline` objects (bundling the `TfidfVectorizer` and the tuned estimators) and saved as `.joblib` files in the `models/trained/` directory. The XGBoost pipeline is now fully prepared to be loaded into the `api/app.py` script for real-time inference without requiring any manual text vectorization steps.